In [0]:
dbutils.widgets.text("p_batch_id", "")
v_batch_id = dbutils.widgets.get("p_batch_id")

In [0]:
%run "../0_common/01. env-config"

In [0]:
%run "../0_common/03.silver-helpers"

In [0]:
bronze_table = f"{catalog_name}.{bronze_schema}.constructors"
silver_table = f"{catalog_name}.{silver_schema}.constructors"

In [0]:
const_df = spark.table(bronze_table).filter(F.col("batch_id") == v_batch_id)
display(const_df.head(5))

In [0]:
const_df = const_df.drop("url")

In [0]:
const_df = const_df.withColumnsRenamed(
    {
        "constructorId":"constructor_id",
        "name":"constructor_name"
    }
)

In [0]:
const_df = const_df.dropDuplicates(["constructor_id"])

In [0]:
import pyspark.sql.functions as F
const_df = const_df.withColumn(
    "nationality",
    F.initcap(F.col("nationality"))
)

In [0]:
columns_to_update = [col for col in const_df.columns if col not in ["constructor_id"]]
columns_to_update

In [0]:
write_to_silver(
    df=const_df,
    target_table=silver_table,
    columns_to_update=columns_to_update,
    merge_condition="t.constructor_id = s.constructor_id"
)

In [0]:
spark.table(silver_table).limit(5)